# CHUẨN BỊ MÔI TRƯỜNG

## Cài đặt thư viện

## Import libraries

In [36]:
import os
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk import tokenize
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import random
import time
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import shutil
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Kiểm tra GPU

In [37]:
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GPU available: True
GPU name: Tesla P100-PCIE-16GB


## Load Datasets

In [38]:
def load_data(path):
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

BASE = "/kaggle/input/opus-2020-englishvietnamese-parallel-corpus"

en_data = load_data(f"{BASE}/TED2020.en-vi.en")
vi_data = load_data(f"{BASE}/TED2020.en-vi.vi")

print(len(en_data), len(vi_data))
print(en_data[:3])
print(vi_data[:3])


323292 323292
['Thank you so much, Chris.', "And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.", 'I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night.']
['Cám ơn rất nhiều, Chris.', 'Đây thật sự là một vinh hạnh lớn cho tôi khi có cơ hội được đứng trên sân khấu này hai lần; Tôi thật sự rất cảm kích.', 'Tôi thực sự bị choáng ngợp bởi hội nghị này, và tôi muốn cám ơn tất cả các bạn vì rất nhiều nhận xét tốt đẹp về những gì tôi đã trình bày đêm hôm trước.']


# Khám phá dữ liệu

In [39]:
def analyze_data(en_data, vi_data):
  en_lengths = [len(s.split()) for s in en_data]
  vi_lengths = [len(s.split()) for s in vi_data]
  print(f"Number of en: {len(en_data)}")
  print(f"Number of vi: {len(vi_data)}")
  print(f"Min length of en: {min(en_lengths)}")
  print(f"Max length of en: {max(en_lengths)}")
  print(f"Min length of vi: {min(vi_lengths)}")
  print(f"Max length of vi: {max(vi_lengths)}")

analyze_data(en_data, vi_data)

Number of en: 323292
Number of vi: 323292
Min length of en: 1
Max length of en: 625
Min length of vi: 1
Max length of vi: 838


# Data preprocessing

## Text clean

In [40]:
def clean_text(text):
  # lower case
  text = text.lower()
  text = re.sub(r"\(.*?\)", "", text)
  # Remove extra space
  text = re.sub(r"\s+", " ", text).strip()
  return text

print("Cleaning data: ")
en_data_clean = [clean_text(s) for s in tqdm(en_data)]
vi_data_clean = [clean_text(s) for s in tqdm(vi_data) ]




Cleaning data: 


100%|██████████| 323292/323292 [00:03<00:00, 93036.06it/s]


## Loaị bỏ câu quá dài hoặc quá ngắn

In [41]:
def filter_sentence(en_data, vi_data, min_len = 3, max_len = 80):
  filter_en = []
  filter_vi = []
  for en, vi in zip(en_data, vi_data):
    en_len = len(en.split())
    vi_len = len(vi.split())
    if(min_len <= en_len <= max_len and min_len <= vi_len <= max_len):
      filter_en.append(en)
      filter_vi.append(vi)
  return filter_en, filter_vi

In [42]:
print(f"Trước khi filter: {len(en_data_clean)} câu ")
print(f"Trước khi filter: {len(vi_data_clean)} câu ")
en_data_clean, vi_data_clean = filter_sentence(en_data_clean, vi_data_clean)
print(f"Sau khi filter: {len(en_data_clean)} câu")
print(f"Sau khi filter: {len(vi_data_clean)} câu")

Trước khi filter: 323292 câu 
Trước khi filter: 323292 câu 
Sau khi filter: 313502 câu
Sau khi filter: 313502 câu


# CHIA TRAIN TEST VAL

In [43]:
random.seed(42)

combined  = list(zip(en_data_clean, vi_data_clean))
random.shuffle(combined)
en_data_clean, vi_data_clean = zip(*combined)

total = len(en_data_clean)
idx_train = int(0.8*total)
idx_val = int(0.9*total)

train_en = en_data_clean[:idx_train]
train_vi = vi_data_clean[:idx_train]

val_en = en_data_clean[idx_train:idx_val]
val_vi = vi_data_clean[idx_train:idx_val]

test_en = en_data_clean[idx_val:]
test_vi = vi_data_clean[idx_val:]

print(f"Train en: {len(train_en)} câu")
print(f"Val en: {len(val_en)} câu")
print(f"Test en: {len(test_en)} câu")

print("\n")
print(f"Train vi: {len(train_vi)} câu")
print(f"Val vi: {len(val_vi)} câu")
print(f"Test vi: {len(test_vi)} câu")




Train en: 250801 câu
Val en: 31350 câu
Test en: 31351 câu


Train vi: 250801 câu
Val vi: 31350 câu
Test vi: 31351 câu


# XÂY DỰNG VOCABULARY VÀ TOKENIZER

In [44]:
class Vocabulary:
  def __init__(self, freq_threshold = 2):
    self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
    self.word2idx = {"<PAD>" : 0, "<SOS>" : 1, "<EOS>" : 2, "<UNK>" : 3}
    self.n_words = 4

  def add_sentence(self, sentence):
    for word in sentence.split():
      self.add_word(word)

  def add_word(self, word):
    if word not in self.word2idx:
      self.word2idx[word] = self.n_words
      self.idx2word[self.n_words] = word
      self.n_words += 1

  def numericalize(self, word):
    tokenized_text = tokenize.word_tokenize(word)
    return [self.word2idx[word] if word in self.word2idx else self.word2idx["<UNK>"] for word in tokenized_text]

en_vocab = Vocabulary(freq_threshold=2)
vi_vocab = Vocabulary(freq_threshold=2)

for en, vi in zip(train_en, train_vi):
  en_vocab.add_sentence(en)
  vi_vocab.add_sentence(vi)


print(f"Unique words in English vocabulary: {en_vocab.n_words}")
print(f"Unique words in Vietnamese vocabulary: {vi_vocab.n_words}")


Unique words in English vocabulary: 132547
Unique words in Vietnamese vocabulary: 67231


In [45]:
class TranslationDataset(Dataset):
  def __init__(self, en_data, vi_data, en_vocab, vi_vocab, max_len = 80):
    self.en_data = en_data
    self.vi_data = vi_data
    self.en_vocab = en_vocab
    self.vi_vocab = vi_vocab
    self.max_len = max_len

  def __len__(self):
    return len(self.en_data)

  def __getitem__(self, idx):
    en = self.en_data[idx]
    vi = self.vi_data[idx]

    en_numericalize = [self.en_vocab.word2idx["<SOS>"]] + \
                      self.en_vocab.numericalize(en) + \
                      [self.en_vocab.word2idx["<EOS>"]]

    vi_numericalize = [self.vi_vocab.word2idx["<SOS>"]] + \
                      self.vi_vocab.numericalize(vi) + \
                      [self.vi_vocab.word2idx["<EOS>"]]

    return torch.tensor(en_numericalize), torch.tensor(vi_numericalize)

In [46]:
def collate_fn(batch):
  en_batch, vi_batch = [], []
  for en, vi in batch:
    en_batch.append(en)
    vi_batch.append(vi)

  en_batch = nn.utils.rnn.pad_sequence(en_batch, batch_first=True, padding_value=0)
  vi_batch = nn.utils.rnn.pad_sequence(vi_batch, batch_first=True, padding_value=0)
  return en_batch, vi_batch

In [47]:
train_datasets = TranslationDataset(train_en, train_vi, en_vocab, vi_vocab)
val_datasets = TranslationDataset(val_en, val_vi, en_vocab, vi_vocab)
test_datasets = TranslationDataset(test_en, test_vi, en_vocab, vi_vocab)

BATCH_SIZE = 128

train_loader = DataLoader(train_datasets, batch_size = BATCH_SIZE, shuffle = True, collate_fn = collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_datasets, batch_size = BATCH_SIZE, shuffle = False, collate_fn = collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_datasets, batch_size = BATCH_SIZE, shuffle = False, collate_fn = collate_fn, num_workers=2, pin_memory=True)

print(f"Train loader: {len(train_loader)} batch")
print(f"Val loader: {len(val_loader)} batch")
print(f"Test loader: {len(test_loader)} batch")

Train loader: 1960 batch
Val loader: 245 batch
Test loader: 245 batch


# MÔ HÌNH SEQ2SEQ VỚI ATTENTION

In [48]:
class Encoder(nn.Module):
  def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, emb_dim)
    self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
    self.dropout = nn.Dropout(dropout)

  def forward(self, src):
    embedded = self.dropout(self.embedding(src))
    outputs, (hidden, cell) = self.rnn(embedded)
    return outputs, hidden, cell




In [49]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

In [50]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(hid_dim + emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim * 2 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))

        a = self.attention(hidden[-1], encoder_outputs)
        a = a.unsqueeze(1)

        weighted = torch.bmm(a, encoder_outputs)
        rnn_input = torch.cat((embedded, weighted), dim=2)

        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        weighted = weighted.squeeze(1)

        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))

        return prediction, hidden, cell

In [51]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

# KHỞI TẠO VÀ TRAIN SEQ2SEQ

In [55]:
model_dir = '/kaggle/working/translation_models'
os.makedirs(model_dir, exist_ok=True)

INPUT_DIM = en_vocab.n_words
OUTPUT_DIM = vi_vocab.n_words
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

# Khởi tạo model
attn = Attention(HID_DIM)
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT, attn)
seq2seq_model = Seq2Seq(enc, dec, device).to(device)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Seq2Seq có {count_parameters(seq2seq_model):,} tham số')

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, 
                             patience=2, min_lr=1e-6)
criterion = nn.CrossEntropyLoss(ignore_index=0)


scaler = GradScaler()

def train_epoch(model, iterator, optimizer, criterion, clip, scaler):
    model.train()
    epoch_loss = 0
    
    for src, trg in tqdm(iterator, desc="Training", leave=False):
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision
        with autocast():
            output = model(src, trg)
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            loss = criterion(output, trg)
        
        # Backward với gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in tqdm(iterator, desc="Validating", leave=False):
            src, trg = src.to(device), trg.to(device)
            
            # Mixed precision cho evaluation
            with autocast():
                output = model(src, trg, 0)
                output_dim = output.shape[-1]
                output = output[:, 1:].reshape(-1, output_dim)
                trg = trg[:, 1:].reshape(-1)
                loss = criterion(output, trg)
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

torch.save({
    'en_vocab': en_vocab,
    'vi_vocab': vi_vocab,
    'config': {
        'INPUT_DIM': INPUT_DIM,
        'OUTPUT_DIM': OUTPUT_DIM,
        'ENC_EMB_DIM': ENC_EMB_DIM,
        'DEC_EMB_DIM': DEC_EMB_DIM,
        'HID_DIM': HID_DIM,
        'N_LAYERS': N_LAYERS,
        'DROPOUT': DROPOUT
    }
}, f'{model_dir}/vocab_and_config.pt')

print(f"Đã lưu vocabulary vào {model_dir}/vocab_and_config.pt")

N_EPOCHS = 10
CLIP = 1
PATIENCE = 3  # Early stopping

best_valid_loss = float('inf')
patience_counter = 0
total_start_time = time.time()


for epoch in range(N_EPOCHS):
    epoch_start_time = time.time()
    print(f'\nEpoch {epoch+1}/{N_EPOCHS}')

    
    # Training
    train_loss = train_epoch(seq2seq_model, train_loader, optimizer, 
                            criterion, CLIP, scaler)
    valid_loss = evaluate(seq2seq_model, val_loader, criterion)
    
    # Update learning rate
    scheduler.step(valid_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Lưu checkpoint mỗi epoch
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': seq2seq_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'train_loss': train_loss,
        'valid_loss': valid_loss,
        'best_valid_loss': best_valid_loss
    }
    
    # Chỉ lưu checkpoint mới nhất 
    torch.save(checkpoint, f'{model_dir}/seq2seq_last_checkpoint.pt')
    
    # Lưu best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        patience_counter = 0
        
        torch.save(seq2seq_model.state_dict(), f'{model_dir}/seq2seq_best.pt')
        torch.save(checkpoint, f'{model_dir}/seq2seq_best_checkpoint.pt')
        print(f'✓ Đã lưu best model (valid_loss: {valid_loss:.4f})')
    else:
        patience_counter += 1
        print(f'No improvement for {patience_counter} epoch(s)')
    
    # Tính thời gian
    epoch_time = time.time() - epoch_start_time
    total_time = time.time() - total_start_time
    remaining = (N_EPOCHS - epoch - 1) * epoch_time
    
    # In kết quả
    print(f'\nTrain Loss: {train_loss:.4f} | Train PPL: {np.exp(train_loss):7.3f}')
    print(f'Val. Loss: {valid_loss:.4f} | Val. PPL: {np.exp(valid_loss):7.3f}')
    print(f'Best Val. Loss: {best_valid_loss:.4f}')
    print(f'poch: {epoch_time/60:.1f}m | Total: {total_time/60:.1f}m | Remaining: {remaining/60:.1f}m')
    print(f'Learning Rate: {current_lr:.6f}')
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping triggered! No improvement for {PATIENCE} epochs.')
        break

# Lưu model cuối cùng


torch.save(seq2seq_model.state_dict(), f'{model_dir}/seq2seq_final.pt')
torch.save({
    'epoch': epoch + 1,
    'model_state_dict': seq2seq_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_loss': train_loss,
    'valid_loss': valid_loss,
    'best_valid_loss': best_valid_loss
}, f'{model_dir}/seq2seq_final_checkpoint.pt')

total_training_time = time.time() - total_start_time
print(f'\nĐã lưu tất cả model vào: {model_dir}')
print(f'Tổng thời gian training: {total_training_time/60:.1f} phút')
print(f'Best validation loss: {best_valid_loss:.4f}')


shutil.copytree(model_dir, '/kaggle/output/translation_models', dirs_exist_ok=True)
print(f'Đã copy models sang /kaggle/output/')


Seq2Seq có 146,196,383 tham số


/tmp/ipykernel_55/1987576598.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Đã lưu vocabulary vào /kaggle/working/translation_models/vocab_and_config.pt

Epoch 1/10


Training:   0%|          | 0/1960 [00:00<?, ?it/s]/tmp/ipykernel_55/1987576598.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.79 GiB. GPU 0 has a total capacity of 15.89 GiB of which 1.59 GiB is free. Process 4831 has 14.30 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 982.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# MÔ HÌNH TRANSFORMER ĐƠN GIẢN

In [ ]:
# Tạo thư mục lưu model
model_dir = '/kaggle/working/translation_models'
os.makedirs(model_dir, exist_ok=True)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8,
                 num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=512, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, src, tgt):
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        src_padding_mask = (src == 0)
        tgt_padding_mask = (tgt == 0)
        
        src_emb = self.dropout(self.pos_encoder(self.src_embedding(src) * np.sqrt(self.d_model)))
        tgt_emb = self.dropout(self.pos_encoder(self.tgt_embedding(tgt) * np.sqrt(self.d_model)))
        
        output = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        
        return self.fc_out(output)

transformer_model = TransformerModel(
    src_vocab_size=en_vocab.n_words,
    tgt_vocab_size=vi_vocab.n_words,
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1
).to(device)



def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nTransformer có {count_parameters(transformer_model):,} tham số')

optimizer_tf = optim.Adam(transformer_model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
scheduler_tf = ReduceLROnPlateau(optimizer_tf, mode='min', factor=0.5, 
                                 patience=2, min_lr=1e-6)
criterion_tf = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)  # Label smoothing

# Mixed Precision
scaler_tf = GradScaler()

def train_transformer(model, iterator, optimizer, criterion, scaler):
    model.train()
    epoch_loss = 0
    
    for src, trg in tqdm(iterator, desc="Training", leave=False):
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        
        trg_input = trg[:, :-1]
        trg_output = trg[:, 1:]
        
        # Mixed precision
        with autocast():
            output = model(src, trg_input)
            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            trg_output = trg_output.reshape(-1)
            loss = criterion(output, trg_output)
        
        # Backward với gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

def evaluate_transformer(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in tqdm(iterator, desc="Validating", leave=False):
            src, trg = src.to(device), trg.to(device)
            
            trg_input = trg[:, :-1]
            trg_output = trg[:, 1:]
            
            # Mixed precision
            with autocast():
                output = model(src, trg_input)
                output_dim = output.shape[-1]
                output = output.reshape(-1, output_dim)
                trg_output = trg_output.reshape(-1)
                loss = criterion(output, trg_output)
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

torch.save({
    'en_vocab': en_vocab,
    'vi_vocab': vi_vocab,
    'transformer_config': {
        'src_vocab_size': en_vocab.n_words,
        'tgt_vocab_size': vi_vocab.n_words,
        'd_model': 256,
        'nhead': 8,
        'num_encoder_layers': 3,
        'num_decoder_layers': 3,
        'dim_feedforward': 512,
        'dropout': 0.1
    }
}, f'{model_dir}/transformer_vocab_and_config.pt')

print(f"Đã lưu Transformer vocabulary và config")


N_EPOCHS_TF = 10
PATIENCE = 3  # Early stopping

best_valid_loss_tf = float('inf')
patience_counter = 0
total_start_time = time.time()




for epoch in range(N_EPOCHS_TF):
    epoch_start_time = time.time()
    print(f'\nTransformer - Epoch {epoch+1}/{N_EPOCHS_TF}')
    
    # Training
    train_loss = train_transformer(transformer_model, train_loader, 
                                   optimizer_tf, criterion_tf, scaler_tf)
    valid_loss = evaluate_transformer(transformer_model, val_loader, criterion_tf)
    
    # Update learning rate
    scheduler_tf.step(valid_loss)
    current_lr = optimizer_tf.param_groups[0]['lr']
    
    # Lưu checkpoint mới nhất
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': transformer_model.state_dict(),
        'optimizer_state_dict': optimizer_tf.state_dict(),
        'scheduler_state_dict': scheduler_tf.state_dict(),
        'scaler_state_dict': scaler_tf.state_dict(),
        'train_loss': train_loss,
        'valid_loss': valid_loss,
        'best_valid_loss': best_valid_loss_tf
    }
    
    # Chỉ lưu checkpoint mới nhất (tiết kiệm dung lượng)
    torch.save(checkpoint, f'{model_dir}/transformer_last_checkpoint.pt')
    
    # Lưu best model
    if valid_loss < best_valid_loss_tf:
        best_valid_loss_tf = valid_loss
        patience_counter = 0
        
        torch.save(transformer_model.state_dict(), f'{model_dir}/transformer_best.pt')
        torch.save(checkpoint, f'{model_dir}/transformer_best_checkpoint.pt')
        print(f'Đã lưu best model (valid_loss: {valid_loss:.4f})')
    else:
        patience_counter += 1
        print(f'No improvement for {patience_counter} epoch(s)')
    
    # Tính thời gian
    epoch_time = time.time() - epoch_start_time
    total_time = time.time() - total_start_time
    remaining = (N_EPOCHS_TF - epoch - 1) * epoch_time
    
    # In kết quả
    print(f'\nTrain Loss: {train_loss:.4f} | Train PPL: {np.exp(train_loss):7.3f}')
    print(f'Val. Loss: {valid_loss:.4f} | Val. PPL: {np.exp(valid_loss):7.3f}')
    print(f'Best Val. Loss: {best_valid_loss_tf:.4f}')
    print(f'Epoch: {epoch_time/60:.1f}m | Total: {total_time/60:.1f}m | Remaining: {remaining/60:.1f}m')
    print(f'Learning Rate: {current_lr:.6f}')
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping triggered! No improvement for {PATIENCE} epochs.')
        break


torch.save(transformer_model.state_dict(), f'{model_dir}/transformer_final.pt')
torch.save({
    'epoch': epoch + 1,
    'model_state_dict': transformer_model.state_dict(),
    'optimizer_state_dict': optimizer_tf.state_dict(),
    'scheduler_state_dict': scheduler_tf.state_dict(),
    'train_loss': train_loss,
    'valid_loss': valid_loss,
    'best_valid_loss': best_valid_loss_tf
}, f'{model_dir}/transformer_final_checkpoint.pt')

total_training_time = time.time() - total_start_time

print(f'\nĐã lưu tất cả model vào: {model_dir}')
print(f'Tổng thời gian training: {total_training_time/60:.1f} phút')
print(f'Best validation loss: {best_valid_loss_tf:.4f}')

# Copy sang output

shutil.copytree(model_dir, '/kaggle/output/translation_models', dirs_exist_ok=True)
print(f'Đã copy models sang /kaggle/output/')

# In danh sách files
print(f'\nFiles đã lưu:')
for file in os.listdir(model_dir):
    size = os.path.getsize(f'{model_dir}/{file}') / (1024*1024)
    print(f'   • {file}: {size:.1f} MB')

# HÀM DỊCH VÀ TEST

In [ ]:
def translate_sentence_seq2seq(sentence, model, src_vocab, tgt_vocab, device, max_len=80):
    model.eval()

    tokens = [src_vocab.word2idx["<SOS>"]] + src_vocab.numericalize(sentence) + [src_vocab.word2idx["<EOS>"]]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

    trg_indexes = [tgt_vocab.word2idx["<SOS>"]]

    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell, encoder_outputs)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == tgt_vocab.word2idx["<EOS>"]:
            break

    trg_tokens = [tgt_vocab.idx2word[i] for i in trg_indexes]
    return ' '.join(trg_tokens[1:-1])

def translate_sentence_transformer(sentence, model, src_vocab, tgt_vocab, device, max_len=80):
    model.eval()

    tokens = [src_vocab.word2idx["<SOS>"]] + src_vocab.numericalize(sentence) + [src_vocab.word2idx["<EOS>"]]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)

    trg_indexes = [tgt_vocab.word2idx["<SOS>"]]

    for _ in range(max_len):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(src_tensor, trg_tensor)

        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == tgt_vocab.word2idx["<EOS>"]:
            break

    trg_tokens = [tgt_vocab.idx2word[i] for i in trg_indexes]
    return ' '.join(trg_tokens[1:-1])




# Test

In [ ]:
print("\nTEST DỊCH ")
test_sentences = [
    "hello, how are you?",
    "i love machine learning",
    "artificial intelligence is amazing"
]

print("\nSeq2Seq với Attention:")
for sent in test_sentences:
    translation = translate_sentence_seq2seq(sent, seq2seq_model, en_vocab, vi_vocab, device)
    print(f"EN: {sent}")
    print(f"VI: {translation}\n")

print("\nTransformer:")
for sent in test_sentences:
    translation = translate_sentence_transformer(sent, transformer_model, en_vocab, vi_vocab, device)
    print(f"EN: {sent}")
    print(f"VI: {translation}\n")